# Proyecto Final: Agente de Ajuste Semafórico
## Identificación de Intersecciones Críticas en Madrid

**Curso:** Data Analytics for Urban Systems
**Módulo:** 4 — Proyecto Final
**Problema:** Identificar los sensores (que representan intersecciones semaforizadas) que tienen una congestión anormalmente alta fuera de las horas pico tradicionales. Estos sensores son candidatos a una re-programación de sus ciclos semafóricos.

### Lógica del Pipeline
1. El **Agente Analista** usará código Python para filtrar el dataset y encontrar los "Top 5" sensores más congestionados entre las 10:00 y las 16:00.
2. El **Agente Auditor** revisará la lógica del Analista y evaluará un posible **sesgo geográfico** (¿están todos los semáforos malos en el centro o hay distribución equitativa?).

## 1. Configuración e Instalación

In [17]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================================

!pip install -q crewai crewai-tools langchain-openai mcp pandas matplotlib seaborn scipy

import os
import getpass
import asyncio
from crewai import Agent, Task, Crew, Process, LLM

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Introduce tu OpenAI API Key: ")

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive', force_remount=True)
except ImportError:
    pass

# Rutas del proyecto
BASE_DIR = '/content/drive/MyDrive/Datos_EMT_Sensores_Trafico_Madrid/'
DATASET_PATH = BASE_DIR + 'resultados_tarea_8/tarea8_dataset_limpio.parquet'
OUTPUT_DIR = BASE_DIR + 'proyecto_final/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Entorno configurado. Dataset listo en:", DATASET_PATH)

✅ Entorno configurado. Dataset listo en: /content/drive/MyDrive/Datos_EMT_Sensores_Trafico_Madrid/resultados_tarea_8/tarea8_dataset_limpio.parquet


## 2. Servidor MCP: Consulta de Sensores

El servidor MCP expondrá una herramienta para que el agente pueda ver un resumen de los datos sin necesidad de cargar todo el Parquet en su memoria de contexto.

In [19]:
# ============================================================
# SERVIDOR MCP PARA EL PROYECTO
# ============================================================

server_code = f"""
from mcp.server import MCPServer
import pandas as pd

mcp = MCPServer("MadridTrafficServer")
DATASET = "{DATASET_PATH}"

@mcp.tool()
def obtener_resumen_sensores() -> str:
    \"\"\"Devuelve una lista con los 10 sensores con mayor volumen promedio histórico.\"\"\"
    try:
        df = pd.read_parquet(DATASET)
        resumen = df.groupby('sensor_id').agg(
            volumen_medio=('volume', 'mean'),
            congestion_media=('congestion', 'mean')
        ).sort_values('volumen_medio', ascending=False).head(10)
        return resumen.to_string()
    except Exception as e:
        return f"Error al leer el archivo: {{str(e)}}"

@mcp.tool()
def obtener_esquema_columnas() -> str:
    \"\"\"Devuelve los nombres de las columnas del dataset.\"\"\"
    try:
        df = pd.read_parquet(DATASET)
        return str(df.columns.tolist())
    except Exception as e:
        return f"Error: {{str(e)}}"

if __name__ == "__main__":
    mcp.run()
"""

with open("mcp_server.py", "w") as f:
    f.write(server_code)

print("✅ Servidor MCP 'mcp_server.py' creado.")

✅ Servidor MCP 'mcp_server.py' creado.


## 3. Configuración del Equipo Multi-Agente
- **Analista:** Buscará los sensores con congestión inusualmente alta en horario valle.
- **Auditor:** Revisará el código del analista y calculará si los sensores encontrados están sesgados geográficamente hacia el centro.

In [20]:
# ============================================================
# DEFINICIÓN DE AGENTES
# ============================================================
llm_local = LLM(model="gpt-4o-mini", temperature=0)

analista = Agent(
    role='Ingeniero de Tráfico Senior',
    goal='Identificar los 5 sensores (semáforos) más congestionados en horario no-punta (10:00 a 16:00).',
    backstory=(
        "Eres un experto en flujo vehicular. Sabes que si un semáforo está congestionado a las 14:00 (fuera de hora punta), "
        "es porque su ciclo está mal calibrado. Escribe código Python usando Pandas para filtrar el dataset, "
        "extrae la hora del timestamp, filtra la franja 10-16h, y agrupa por 'sensor_id' para sacar el Top 5. "
        "Siempre usa print() para mostrar tus resultados."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm_local
)

auditor = Agent(
    role='Auditor de Equidad Espacial',
    goal='Verificar la lógica del analista y evaluar el sesgo geográfico (Centro vs Periferia) de los sensores seleccionados.',
    backstory=(
        "Eres un planificador urbano enfocado en la equidad. Tu trabajo es revisar el Top 5 del analista. "
        "Debes comprobar si esos semáforos están concentrados en el centro o distribuidos. "
        "Si todos están en el centro (distancia < 5km de la Puerta del Sol), debes alertar sobre el Efecto Lámpara. "
        "Genera un veredicto final claro y ético."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm_local
)

print("✅ Agentes listos.")

✅ Agentes listos.


## 4. Orquestación de Tareas

El Analista ejecutará el código para encontrar los semáforos. El Auditor usará el resultado para evaluar la distribución espacial.

In [21]:
# ============================================================
# DEFINICIÓN DE TAREAS
# ============================================================

task_analisis = Task(
    description=(
        f"Escribe un script en Python que haga lo siguiente:\n"
        f"1. Carga el dataset desde: {DATASET_PATH}\n"
        f"2. Convierte 'timestamp' a datetime y extrae la hora (columna 'hour').\n"
        f"3. Filtra los datos para quedarte SOLO con las horas entre 10 y 16 (inclusive).\n"
        f"4. Agrupa por 'sensor_id' y calcula la congestión media en esa franja.\n"
        f"5. Imprime el Top 5 de sensores con mayor congestión (usa print()).\n"
        f"6. NO ejecutes el código, solo escríbelo en un bloque Markdown."
    ),
    expected_output="Un bloque de código Python con el Top 5 de sensores.",
    agent=analista
)

task_auditoria = Task(
    description=(
        "Revisa el código del analista y sus resultados (los 5 sensores).\n"
        "1. Evalúa si el código tiene Data Leakage o errores lógicos.\n"
        "2. Asume que cada sensor_id corresponde a una intersección en Madrid. "
        "¿La selección de estos 5 sensores podría estar sesgada hacia el centro de la ciudad?\n"
        "3. Redacta un veredicto final (APROBADO / APROBADO CON RESERVAS) y un párrafo explicando la equidad de tu recomendación semafórica."
    ),
    expected_output="Un reporte de auditoría ética y técnica.",
    agent=auditor,
    context=[task_analisis]
)

print("✅ Tareas definidas.")

✅ Tareas definidas.


## 5. Ejecución del Equipo

Iniciamos el Crew para que el Analista busque los datos y el Auditor emita su juicio.

In [23]:
# ============================================================
# EJECUCIÓN DEL CREW
# ============================================================

proyecto_crew = Crew(
    agents=[analista, auditor],
    tasks=[task_analisis, task_auditoria],
    process=Process.sequential,
    verbose=True
)

print("🚀 Iniciando la identificación de semáforos...\n")

try:
    resultado_final = await proyecto_crew.kickoff_async()

    print("\n" + "="*60)
    print("REPORTE FINAL DEL PROYECTO (AUDITORÍA)")
    print("="*60)
    print(resultado_final)

    # Guardar reporte
    with open(f'{OUTPUT_DIR}reporte_semaforos.md', 'w') as f:
        f.write(str(resultado_final))
    print(f"\n✅ Reporte guardado en: {OUTPUT_DIR}reporte_semaforos.md")

except Exception as e:
    print(f"\n❌ Falló la ejecución: {str(e)}")

🚀 Iniciando la identificación de semáforos...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0c40b9cf-8ba9-462e-91c9-d164f9acf90a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Escribe un script en Python que haga lo siguiente:                                                       │
│  1. Carga el dataset desde:                                                                                     │
│  /content/drive/MyDrive/Datos_EMT_Sensores_Trafico_Madrid/resultados_tarea_8/tarea8_dataset_limpio.parquet      │
│  2. Convierte 'timestamp' a datetime y extrae la hora (columna 'hour').                                         │
│  3. Filtra los datos para quedarte SOLO con las horas entre 10 y 16 (inclusive).                                │
│  4. Agrupa por 'sensor_id' y calcula la congestión media en esa franja.                                         │
│  5. Imprime el Top 5 de sensores con mayor congestión (usa print()).                                            │
│  6. NO ejecutes el código, solo escríbelo en un bloque Markdown.                                                │
│  ID: ae1c13d6-7f7c-4f4a-825b-dc845850bbe8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ingeniero de Tráfico Senior                                                                             │
│                                                                                                                 │
│  Task: Escribe un script en Python que haga lo siguiente:                                                       │
│  1. Carga el dataset desde:                                                                                     │
│  /content/drive/MyDrive/Datos_EMT_Sensores_Trafico_Madrid/resultados_tarea_8/tarea8_dataset_limpio.parquet      │
│  2. Convierte 'timestamp' a datetime y extrae la hora (columna 'hour').                                         │
│  3. Filtra los datos para quedarte SOLO con las horas entre 10 y 16 (inclusive).                                │
│  4. Agrupa por 'sensor_id' y calcula la congestión media en esa franja.                                         │
│  5. Imprime el Top 5 de sensores con mayor congestión (usa print()).                                            │
│  6. NO ejecutes el código, solo escríbelo en un bloque Markdown.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Ingeniero de Tráfico Senior                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  import pandas as pd                                                                                            │
│                                                                                                                 │
│  # 1. Carga el dataset                                                                                          │
│  df =                                                                                                           │
│  pd.read_parquet('/content/drive/MyDrive/Datos_EMT_Sensores_Trafico_Madrid/resultados_tarea_8/tarea8_dataset_l  │
│  impio.parquet')                                                                                                │
│                                                                                                                 │
│  # 2. Convierte 'timestamp' a datetime y extrae la hora                                                         │
│  df['timestamp'] = pd.to_datetime(df['timestamp'])                                                              │
│  df['hour'] = df['timestamp'].dt.hour                                                                           │
│                                                                                                                 │
│  # 3. Filtra los datos para quedarte SOLO con las horas entre 10 y 16 (inclusive)                               │
│  filtered_df = df[(df['hour'] >= 10) & (df['hour'] <= 16)]                                                      │
│                                                                                                                 │
│  # 4. Agrupa por 'sensor_id' y calcula la congestión media en esa franja                                        │
│  congestion_mean = filtered_df.groupby('sensor_id')['congestion'].mean().reset_index()                          │
│                                                                                                                 │
│  # 5. Imprime el Top 5 de sensores con mayor congestión                                                         │
│  top_5_sensors = congestion_mean.nlargest(5, 'congestion')                                                      │
│  print(top_5_sensors)                                                                                           │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (4 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Auditor de Equidad Espacial                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Auditoría de Equidad Espacial                                                                              │
│                                                                                                                 │
│  #### 1. Evaluación del Código                                                                                  │
│  El código proporcionado parece estar bien estructurado y no presenta errores lógicos evidentes ni Data         │
│  Leakage. Se carga un dataset, se convierte la columna de timestamp a un formato datetime, se filtran los       │
│  datos para un rango horario específico y se calcula la congestión media por sensor. Sin embargo, es            │
│  importante considerar que el filtrado de datos solo en las horas entre las 10 y las 16 puede no representar    │
│  adecuadamente la congestión total a lo largo del día, lo que podría limitar la generalización de los           │
│  resultados.                                                                                                    │
│                                                                                                                 │
│  #### 2. Sesgo Geográfico                                                                                       │
│  Para determinar si la selección de los 5 sensores está sesgada hacia el centro de la ciudad, es necesario      │
│  conocer la ubicación geográfica de cada sensor. Sin embargo, dado que no se proporciona información sobre las  │
│  coordenadas de los sensores en el código, no puedo realizar un análisis directo. Si todos los sensores         │
│  seleccionados están ubicados a menos de 5 km de la Puerta del Sol, esto indicaría un sesgo hacia el centro de  │
│  la ciudad, lo que podría resultar en el Efecto Lámpara, donde se priorizan áreas céntricas en detrimento de    │
│  la periferia.                                                                                                  │
│                                                                                                                 │
│  #### 3. Veredicto Final                                                                                        │
│  **APROBADO CON RESERVAS**                                                                                      │
│                                                                                                                 │
│  La selección de los 5 sensores debe ser revisada para asegurar que no estén concentrados en el centro de       │
│  Madrid. Es crucial que la planificación urbana y la gestión del tráfico consideren la equidad espacial,        │
│  garantizando que las áreas periféricas también reciban atención adecuada en términos de infraestructura y      │
│  servicios. La congestión en el centro no debe ser la única prioridad, ya que esto podría perpetuar             │
│  desigualdades en el acceso y la calidad de vida en diferentes partes de la ciudad. Se recomienda realizar un   │
│  análisis geográfico de los sensores seleccionados y considerar la inclusión de sensores en áreas periféricas   │
│  para una evaluación más equitativa de la congestión del tráfico en Madrid.                                     │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Revisa el código del analista y sus resultados (los 5 sensores).                                         │
│  1. Evalúa si el código tiene Data Leakage o errores lógicos.                                                   │
│  2. Asume que cada sensor_id corresponde a una intersección en Madrid. ¿La selección de estos 5 sensores        │
│  podría estar sesgada hacia el centro de la ciudad?                                                             │
│  3. Redacta un veredicto final (APROBADO / APROBADO CON RESERVAS) y un párrafo explicando la equidad de tu      │
│  recomendación semafórica.                                                                                      │
│  Agent: Auditor de Equidad Espacial                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0c40b9cf-8ba9-462e-91c9-d164f9acf90a                                                                       │
│  Final Output: ### Auditoría de Equidad Espacial                                                                │
│                                                                                                                 │
│  #### 1. Evaluación del Código                                                                                  │
│  El código proporcionado parece estar bien estructurado y no presenta errores lógicos evidentes ni Data         │
│  Leakage. Se carga un dataset, se convierte la columna de timestamp a un formato datetime, se filtran los       │
│  datos para un rango horario específico y se calcula la congestión media por sensor. Sin embargo, es            │
│  importante considerar que el filtrado de datos solo en las horas entre las 10 y las 16 puede no representar    │
│  adecuadamente la congestión total a lo largo del día, lo que podría limitar la generalización de los           │
│  resultados.                                                                                                    │
│                                                                                                                 │
│  #### 2. Sesgo Geográfico                                                                                       │
│  Para determinar si la selección de los 5 sensores está sesgada hacia el centro de la ciudad, es necesario      │
│  conocer la ubicación geográfica de cada sensor. Sin embargo, dado que no se proporciona información sobre las  │
│  coordenadas de los sensores en el código, no puedo realizar un análisis directo. Si todos los sensores         │
│  seleccionados están ubicados a menos de 5 km de la Puerta del Sol, esto indicaría un sesgo hacia el centro de  │
│  la ciudad, lo que podría resultar en el Efecto Lámpara, donde se priorizan áreas céntricas en detrimento de    │
│  la periferia.                                                                                                  │
│                                                                                                                 │
│  #### 3. Veredicto Final                                                                                        │
│  **APROBADO CON RESERVAS**                                                                                      │
│                                                                                                                 │
│  La selección de los 5 sensores debe ser revisada para asegurar que no estén concentrados en el centro de       │
│  Madrid. Es crucial que la planificación urbana y la gestión del tráfico consideren la equidad espacial,        │
│  garantizando que las áreas periféricas también reciban atención adecuada en términos de infraestructura y      │
│  servicios. La congestión en el centro no debe ser la única prioridad, ya que esto podría perpetuar             │
│  desigualdades en el acceso y la calidad de vida en diferentes partes de la ciudad. Se recomienda realizar un   │
│  análisis geográfico de los sensores seleccionados y considerar la inclusión de sensores en áreas periféricas   │
│  para una evaluación más equitativa de la congestión del tráfico en Madrid.                                     │
│                                                                                                                 │
│                                                       


REPORTE FINAL DEL PROYECTO (AUDITORÍA)
### Auditoría de Equidad Espacial

#### 1. Evaluación del Código
El código proporcionado parece estar bien estructurado y no presenta errores lógicos evidentes ni Data Leakage. Se carga un dataset, se convierte la columna de timestamp a un formato datetime, se filtran los datos para un rango horario específico y se calcula la congestión media por sensor. Sin embargo, es importante considerar que el filtrado de datos solo en las horas entre las 10 y las 16 puede no representar adecuadamente la congestión total a lo largo del día, lo que podría limitar la generalización de los resultados.

#### 2. Sesgo Geográfico
Para determinar si la selección de los 5 sensores está sesgada hacia el centro de la ciudad, es necesario conocer la ubicación geográfica de cada sensor. Sin embargo, dado que no se proporciona información sobre las coordenadas de los sensores en el código, no puedo realizar un análisis directo. Si todos los sensores seleccionados están u

## 6. Auditoría Ética Cuantitativa

El Auditor generó un juicio cualitativo. Ahora, como equipo humano, deben demostrar numéricamente si el Agente Analista eligió semáforos sesgados geográficamente.

> **TODO:** Escribir el código que cruce el Top 5 del analista con las coordenadas y calcule su distancia al centro.

In [24]:
# ============================================================
# CÁLCULO DE SESGO (FAIRNESS) - PIPELINE AUTÓNOMO CON DATOS ZENODO
# ============================================================
import pandas as pd
import numpy as np
import h5py
import os
import requests

print("🚀 Iniciando pipeline autónomo de auditoría ética...")

# --- 1. Descargar fixed_sensor_data.h5 si no está en la sesión ---
ruta_trafico = "/content/fixed_sensor_data.h5"
if not os.path.exists(ruta_trafico):
    print("Descargando fixed_sensor_data.h5 desde Zenodo (194 MB)...")
    url_trafico = "https://zenodo.org/records/18781181/files/fixed_sensor_data.h5?download=1"
    r = requests.get(url_trafico, stream=True)
    with open(ruta_trafico, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("✅ Descarga de tráfico completada.")

# --- 2. Calcular el Top 5 de sensores en horario valle (10-16h) ---
print("\nCalculando los 5 sensores más congestionados entre las 10:00 y 16:00...")
with h5py.File(ruta_trafico, 'r') as f:
    features = [feat.decode('utf-8') for feat in f['metadata/features'][:]]
    nodes = f['metadata/nodes'][:]
    times = f['metadata/times'][:]
    idx_conges = features.index('conges_levl')

    # Extraer datos completos
    conges_data = f['data/values'][:, :, idx_conges]
    times_dt = pd.to_datetime([t.decode('utf-8') for t in times])
    horas = times_dt.hour

# Filtrar franja 10-16h
mask_valle = (horas >= 10) & (horas <= 16)
conges_valle = conges_data[mask_valle, :]

# Calcular la media por sensor (ignorando NaNs)
conges_media = np.nanmean(conges_valle, axis=0)

# Crear DataFrame temporal para encontrar el Top 5
df_conges = pd.DataFrame({'sensor_id': nodes, 'congestion_media': conges_media})
top_5 = df_conges.nlargest(5, 'congestion_media')
top_sensores_llm = top_5['sensor_id'].tolist()

print(f"✅ Top 5 Sensores identificados: {top_sensores_llm}")
print(f"   Congestión media en valle:\n{top_5.to_string(index=False)}")

# --- 3. Descargar road_info.h5 si no está en la sesión ---
ruta_road = "/content/road_info.h5"
if not os.path.exists(ruta_road):
    print("\nDescargando road_info.h5 desde Zenodo (8.6 MB)...")
    url_road = "https://zenodo.org/records/18781181/files/road_info.h5?download=1"
    r = requests.get(url_road)
    with open(ruta_road, 'wb') as f:
        f.write(r.content)
    print("✅ Descarga de coordenadas completada.")

# --- 4. Auditoría Ética (Cálculo de distancia) ---
print("\nEjecutando auditoría ética (Distancia al centro)...")
with h5py.File(ruta_road, 'r') as f:
    features_road = [feat.decode('utf-8') for feat in f['metadata/features'][:]]
    nodes_road = f['metadata/nodes'][:]
    values_road = f['data/values'][:]

    x_coords = values_road[:, features_road.index('x')]
    y_coords = values_road[:, features_road.index('y')]

df_coords = pd.DataFrame({'sensor_id': nodes_road, 'x': x_coords, 'y': y_coords})
df_top = df_coords[df_coords['sensor_id'].isin(top_sensores_llm)].copy()

# Calcular distancia al centro UTM (440000, 4474000)
df_top['dist_centro_km'] = np.sqrt((df_top['x'] - 440000)**2 + (df_top['y'] - 4474000)**2) / 1000

print("\n📍 Ubicación geográfica de los semáforos a ajustar:")
print(df_top[['sensor_id', 'x', 'y', 'dist_centro_km']].to_string(index=False))

dist_media = df_top['dist_centro_km'].mean()
print(f"\nDistancia media al centro de Madrid: {dist_media:.2f} km")

if dist_media < 5:
    print("\n⚠️ ALERTA ÉTICA (Efecto Lámpara):")
    print("   Los semáforos recomendados están fuertemente concentrados en el centro.")
    print("   El algoritmo está ignorando la periferia. Se requiere una revisión de los datos.")
else:
    print("\n✅ EQUIDAD GEOGRÁFICA:")
    print("   Los sensores recomendados están distribuidos equitativamente en la ciudad.")

🚀 Iniciando pipeline autónomo de auditoría ética...

Calculando los 5 sensores más congestionados entre las 10:00 y 16:00...
✅ Top 5 Sensores identificados: [17726, 64717, 59088, 10923, 13675]
   Congestión media en valle:
 sensor_id  congestion_media
     17726         60.114754
     64717         56.975723
     59088         55.031106
     10923         54.767281
     13675         54.321799

Ejecutando auditoría ética (Distancia al centro)...

📍 Ubicación geográfica de los semáforos a ajustar:
 sensor_id             x            y  dist_centro_km
     10923 439923.814371 4.474730e+06        0.733698
     13675 442916.838773 4.474678e+06        2.994630
     17726 446965.681949 4.470590e+06        7.755750
     59088 445041.142356 4.470719e+06        6.015056
     64717 440807.704317 4.471237e+06        2.878582

Distancia media al centro de Madrid: 4.08 km

⚠️ ALERTA ÉTICA (Efecto Lámpara):
   Los semáforos recomendados están fuertemente concentrados en el centro.
   El algoritmo es